# Sesión 6 · Notebook 2 — Consultar logs en Elasticsearch desde Python

Los logs que Logstash indexó en la Sesión 3 también se pueden leer con código.

El índice es `orderflow-logs-*` y contiene **los dos orígenes**: el JSON del
`order-processor` y el texto del `order-generator` que grok estructuró. Se
distinguen por el campo `tags`.

**Antes de empezar:** `pip install -r notebooks/requirements.txt`.

In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")
es.info()["version"]["number"]   # confirma la conexión

## Traer los últimos logs de nivel ERROR

In [ ]:
resp = es.search(
    index="orderflow-logs-*",
    size=10,
    sort=[{"@timestamp": {"order": "desc"}}],
    query={"term": {"level.keyword": "ERROR"}},
)

print("total de coincidencias:", resp["hits"]["total"]["value"])
for hit in resp["hits"]["hits"][:5]:
    src = hit["_source"]
    print(src.get("@timestamp"), "|", src.get("level"), "|", src.get("message"))

### Por qué `level.keyword` y no `level`

Elasticsearch indexa cada campo de texto dos veces: como `text` (troceado en
palabras, para buscar) y como `keyword` (el valor entero, para agrupar y filtrar
de forma exacta).

Un `term` sobre `level` puede fallar, porque busca el valor exacto contra un
campo que fue troceado. Sobre `level.keyword` funciona siempre.

Es la misma idea que en Kibana, donde para agrupar te ofrece `reason.keyword` y
no `reason`.

## Agregación: contar sin traer documentos

En vez de traerse mil documentos a Python para contarlos, se le pide a
Elasticsearch que agrupe y cuente él. Es órdenes de magnitud más barato: viaja un
resumen por la red, no los datos.

In [ ]:
resp = es.search(
    index="orderflow-logs-*",
    size=0,   # no queremos documentos, solo la agregación
    aggs={"por_nivel": {"terms": {"field": "level.keyword"}}},
)

for b in resp["aggregations"]["por_nivel"]["buckets"]:
    print(f"{b['key']:>8} : {b['doc_count']}")

## Los dos orígenes, contados por separado

El campo `tags` es lo que permite distinguirlos. Viene de los `tags` que pusiste
en cada `input` del pipeline de Logstash.

In [ ]:
resp = es.search(
    index="orderflow-logs-*",
    size=0,
    aggs={"por_origen": {"terms": {"field": "tags"}}},
)

for b in resp["aggregations"]["por_origen"]["buckets"]:
    print(f"{b['key']:>12} : {b['doc_count']}")

## Llevar los resultados a un DataFrame

In [ ]:
import pandas as pd

resp = es.search(
    index="orderflow-logs-*",
    size=500,
    sort=[{"@timestamp": {"order": "desc"}}],
    query={"range": {"@timestamp": {"gte": "now-1h"}}},
)

filas = [h["_source"] for h in resp["hits"]["hits"]]
df = pd.DataFrame(filas)
print("dimensiones:", df.shape)

if df.empty:
    print("sin logs en la última hora")
else:
    display(df[["@timestamp", "level", "message"]].head())

## Ejercicio

1. Cuenta cuántos errores hubo **por hora** en las últimas 6 horas, usando una
   agregación `date_histogram` sobre `@timestamp` con
   `calendar_interval: "hour"`.

2. Agrupa las órdenes fallidas por su campo `reason` y ordénalas de mayor a menor.
   Compara el resultado con lo que devuelve esta consulta en Prometheus:

   ```promql
   topk(5, sum by (reason) (orderflow_orders_failed_total))
   ```

   Los dos números salen de sistemas distintos y deberían contar la misma
   historia. Si no coinciden, explica en dos líneas por qué.

   *Pista: los valores posibles de `reason` están en `docs/metricas.md`.
   `db_error` no es uno de ellos, aunque aparezca en algún material antiguo.*